# **0. LIBRERÍAS Y CONFIGURACIÓN**

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Preprocesamiento y Modelado
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.preprocessing import TargetEncoder
# modelos

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

# Métricas
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# **1. Carga de datos procesados y exploración inicial**

In [174]:
df = pd.read_parquet("../data/processed/dataset.parquet")



# **Feature engineering**

In [175]:
# Filtro de precios entre 500k y 50M, filtro que se había creado en el EDA toca volver a crearlo aquí para que el modelo no se vea afectado por outliers
df_filtrado = df[(df["Precio"] >= 500_000) & (df["Precio"] <= 50_000_000) & (df["Área Construida (m2)"] <= 2500) & (df["Área Privada (m2)"] <= 2500)].copy()


In [176]:
# Definimos nuevamente piso_cat que quedó en el EDA

def piso_cat(x):
    if pd.isna(x):
        return "missing"
    elif x == 1:
        return "1"
    elif x == 2:
        return "2"
    else:
        return "3+"

df_filtrado["Piso_cat"] = df_filtrado["Piso N°"].apply(piso_cat)


# **Split de datos (train/test)**

### Separación de variables predictoras (X) y variable objetivo (y, "Precio")

### <span style="color: #dc2626;">!!! **Importante: comentarios para recordar**</span>


1. voy a usar Barrio_group como parte de las features, pero la idea es que luego podamos definir realmente cuales van a ser las variables que vamos a usar despues de que toda la limpieza y análisis haya terminado, estoy usando el que hace que vaya a "otros"
2. también estoy usando piso_cat
3. pareciera que para poder usar optuna, es mejor hacer algo como train, test y validation porque optuna corremos el riesgo de ajustar el modelo al test indirectamente, la idea es que  Optuna compare configuraciones sin “mirar” el test final, y eso es algo que no hicimos en el trabajo de la profe camila

70% train: el modelo aprende
15% validation : Optuna prueba distintas combinaciones y decide cuáles son mejores
15% test: una sola vez al final para medir el desempeño real

4. tenemos que definir que métrica nos importa mas para nuestros modelos entre MAE, RMSE y R2


In [177]:
df_filtrado.columns

Index(['ID', 'Barrio', 'Tipo de Inmueble', 'Estado', 'Antigüedad',
       'Área Construida (m2)', 'Área Privada (m2)', 'Estrato', 'Baños',
       'Habitaciones', 'Parqueaderos', 'Piso N°', 'URL', 'Precio',
       'Barrio_clean', 'Barrio_group', 'Piso_cat'],
      dtype='str')

In [178]:
numeric_features = [
    "Área Construida (m2)",
    "Área Privada (m2)",
    "Estrato",
    "Baños",
    "Habitaciones",
    "Parqueaderos",
]

categorical_features = [
    "Tipo de Inmueble",
    "Estado",
    "Antigüedad",
    "Barrio_group",
    "Piso_cat",
]

features = numeric_features + categorical_features



In [179]:
# Separación de variables predictoras (X) y variable objetivo (y, "Precio")

X = df_filtrado[features]
y = df_filtrado["Precio"]

## 70% train, 30% temporal test

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
)  

# Del 30% temporal, mitad validation y mitad test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

# Checamos el shape de las variables de entreno y prueba X, Y
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (3719, 11)
y_train: (3719,)
X_val: (797, 11)
y_val: (797,)
X_test: (798, 11)
y_test: (798,)


## **Pipeline: preprocesamiento + modelo (evitar data leakage)**

In [180]:
df_filtrado.info()

<class 'pandas.DataFrame'>
Index: 5314 entries, 0 to 5627
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    5314 non-null   str    
 1   Barrio                5314 non-null   str    
 2   Tipo de Inmueble      5314 non-null   str    
 3   Estado                5314 non-null   str    
 4   Antigüedad            4032 non-null   str    
 5   Área Construida (m2)  5314 non-null   float64
 6   Área Privada (m2)     5314 non-null   float64
 7   Estrato               5265 non-null   float64
 8   Baños                 5273 non-null   float64
 9   Habitaciones          5226 non-null   float64
 10  Parqueaderos          5314 non-null   int64  
 11  Piso N°               2847 non-null   float64
 12  URL                   5314 non-null   str    
 13  Precio                5314 non-null   int64  
 14  Barrio_clean          5314 non-null   str    
 15  Barrio_group          5314 non-null  

### **Pipelines y transformaciones**

Modelos: 

1. LinearRegression
2. RandomForestRegressor
3. LGBMRegressor

- `PowerTransformer` para normalizar la distribución del Precio, usamos `Yeo-Johnson` porque a pesar de que nuestra variable `Precio` ya tiene valores positivos y podríamos usar `Box-Cox`, nos pareció mejor un metodo que fuera tan estricto, es mas flexible, y aunque ambos buscan redcir la asimetría y estabilizar la varianza, Box-Cox es estricamente mayores a 0, y queríamos una transformación mas segura y fácil de integrar en la pipeline

ademas, es mas flexible que hacer escala logarítimca `np.log1p` sobre nuestra y "precio" porque se adapta a los datos en vez de asumir siempre logartimo.

- PowerTransformer(y)  →  normaliza la distribución del PRECIO 
- RobustScaler(X)      →  escala numeric_features siendo robusto a outliers solo para LinearRegression porque en los de arboles el escalado no influye

Para el modelo lineal (`LinearRegression`) se usa **`RobustScaler`** en lugar de `StandardScaler`.

| Scaler | Fórmula | Problema |
|---|---|---|
| `StandardScaler` | `z = (x − media) / std` | La media y std se ven jaladas por outliers |
| `RobustScaler` | `z = (x − mediana) / IQR` | La mediana y el IQR son resistentes a outliers |

IQR (Rango Intercuartílico)** es la distancia entre el percentil 25 (Q1) y el percentil 75 (Q3).
Cubre el **50% central de los datos**, ignorando los extremos al calcular la escala.

In [181]:
areas = ["Área Construida (m2)", "Área Privada (m2)"]

df_filtrado[areas].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Área Construida (m2),5314.0,106.319407,103.569580,1.0,20.0,35.0,60.0,80.0,115.0,280.0,480.0,2416.0
Área Privada (m2),5314.0,106.755237,106.914379,1.0,20.0,35.0,60.0,79.0,115.0,280.0,500.0,2416.0


<span style="color: orange;"><strong>Observación actualizada</strong></span>

`Área Privada` aún presenta valores máximos muy altos, por lo que el filtro debe aplicarse en ambas variables de área:

```python
df_filtrado = df_filtrado[
    (df_filtrado["Área Construida (m2)"] <= 2500) &
    (df_filtrado["Área Privada (m2)"] <= 2500)
].copy()
```


Aunque el filtro reduce bastante los valores extremos, la distribución de las áreas sigue siendo asimétrica.

Mediana de Área Construida (m2): 80 m²
Media: 106.7 m²
IQR: 115 - 60 = 55 m²

Esto sugiere que todavía hay valores altos que desplazan la media hacia arriba.
Por eso, StandardScaler seguiría centrando con respecto a una media afectada por extremos, mientras que RobustScaler usa la mediana y el rango intercuartílico (IQR), representando mejor el comportamiento típico de los inmuebles.

In [182]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Área Construida (m2)", "Área Privada (m2)")
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Construida (m2)"],
        name="Área Construida",
        marker=dict(color="#0f766e"),
        fillcolor="rgba(15, 118, 110, 0.35)",
        line=dict(color="#0f766e"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área construida: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Privada (m2)"],
        name="Área Privada",
        marker=dict(color="#b45309"),
        fillcolor="rgba(180, 83, 9, 0.35)",
        line=dict(color="#b45309"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área privada: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Distribución y valores extremos en las variables de área",
    template="plotly_white",
    showlegend=False,
    width=1050,
    height=450,
    font=dict(size=13),
    margin=dict(t=70, l=40, r=40, b=40)
)

fig.update_xaxes(title_text="Metros cuadrados", row=1, col=1)
fig.update_xaxes(title_text="Metros cuadrados", row=1, col=2)

fig.show()


In [183]:
# Mira qué hay entre 500 y 4000
df[(df["Área Construida (m2)"] > 500) & 
   (df["Área Construida (m2)"] <= 4000)][["Barrio", "Tipo de Inmueble", "Área Construida (m2)", "Precio"]].sort_values("Área Construida (m2)", ascending=False).head(20)

,Barrio,Tipo de Inmueble,Área Construida (m2),Precio
216,Castilla,Apartamento,4000.0,950000
1483,Buenos aires,Apartamento,4000.0,1150000
255,Pedregal,Apartamento,4000.0,800000
218,Girardot,Apartaestudio,4000.0,1100000
207,Castilla,Apartamento,4000.0,1000000
706,Boston,Apartamento,3900.0,830000
705,Boston,Apartamento,3900.0,830000
704,Boston,Apartamento,3900.0,830000
1330,Buenos aires,Apartamento,3800.0,1200000
2173,Centro,Apartaestudio,3700.0,1100000


In [189]:
df_filtrado['Barrio_group'].value_counts

<bound method IndexOpsMixin.value_counts of 0                   BOSTON
1                 MANRIQUE
2                  ROBLEDO
3               EL POBLADO
4                    OTROS
               ...        
5623        LOMA DEL INDIO
5624                 OTROS
5625              LAURELES
5626    BELEN LAS MERCEDES
5627        BELEN LA PALMA
Name: Barrio_group, Length: 5314, dtype: str>

In [ ]:
# Transformers

# ---------------------------Transformadores---------------------------

numeric_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),  # imputamos faltantes numéricos con la mediana (que ya no deberíamos de tener)
        ("scaler", RobustScaler()),                   # escalado para modelos lineales
    ]
)

numeric_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),  # los árboles no requieren escalado
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", TargetEncoder(smooth=10)),  # importante para barrios con pocas muestras, suaviza la media del target para evitar overfitting, toma el promedio global
    ]
)

# Usamos TargetEncoder porque resume las categorías según su relación con el target ("Precio"),
# y evita expandir demasiado la dimensionalidad cuando hay varias categorías.

# ---------------------------Preprocesadores---------------------------

# ---Modelo lineal---
lr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---Trees: RandomForest y LightGBM---
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_tree, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---------------------------Pipelines---------------------------

lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", lr_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=LinearRegression(),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(random_state=42),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=LGBMRegressor(random_state=42),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)


# **Entrenar: levantamos el MLflow Tracking Server**

```bash
mlflow server \
  --host 127.0.0.1 \
  --port 5001 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns
```
Opción que me funcionó con Powershell
```powershell
mlflow server `
  --host 127.0.0.1 `
  --port 5001 `
  --backend-store-uri sqlite:///mlflow.db `
  --default-artifact-root ./mlruns
```

`Puerto: http://127.0.0.1:5001`

Y se crea el file `mlflow.db`

In [191]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

ImportError: cannot import name 'service' from 'google.protobuf' (c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\google\protobuf\__init__.py)